# 3D Gaussian Splatting Baseline

This notebook uses the extracted 12-view PyBullet frame and calls `gsplat` directly as a reference renderer/trainer. Use this as the baseline to compare against your own from-scratch renderer later.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
drive_project = Path("/content/drive/MyDrive/CS231N_Project")

# Works from either the repo root, the douglas_3dsplat folder, or Google Colab Drive.
if (cwd / "data" / "ping_pong_12view_frame_0000").exists():
    PACKAGE_ROOT = cwd
    REPO_ROOT = cwd.parent
    DATA_DIR = PACKAGE_ROOT / "data" / "ping_pong_12view_frame_0000"
    LOCAL_OUTPUT_DIR = PACKAGE_ROOT / "runs" / "gsplat_baseline_frame_0000"
elif (cwd / "douglas_3dsplat" / "data" / "ping_pong_12view_frame_0000").exists():
    REPO_ROOT = cwd
    PACKAGE_ROOT = cwd / "douglas_3dsplat"
    DATA_DIR = PACKAGE_ROOT / "data" / "ping_pong_12view_frame_0000"
    LOCAL_OUTPUT_DIR = PACKAGE_ROOT / "runs" / "gsplat_baseline_frame_0000"
elif (drive_project / "img_data" / "transforms.json").exists():
    REPO_ROOT = drive_project
    PACKAGE_ROOT = drive_project / "douglas_3dsplat"
    DATA_DIR = drive_project / "img_data"
    LOCAL_OUTPUT_DIR = drive_project / "gaussian_output"
else:
    raise FileNotFoundError(
        "Could not find the 12-view data. Expected one of:\n"
        "  douglas_3dsplat/data/ping_pong_12view_frame_0000/transforms.json\n"
        "  /content/drive/MyDrive/CS231N_Project/img_data/transforms.json"
    )

if PACKAGE_ROOT.exists():
    sys.path.insert(0, str(PACKAGE_ROOT))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("package:", PACKAGE_ROOT, "exists=", PACKAGE_ROOT.exists())
print("data:", DATA_DIR, "exists=", DATA_DIR.exists())
print("output:", LOCAL_OUTPUT_DIR)
print("transforms:", (DATA_DIR / "transforms.json").exists())


## Preview the 12 synchronized views

These images are all from the same simulation frame, seen from 12 calibrated cameras.


In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt

with (DATA_DIR / "transforms.json").open() as f:
    scene = json.load(f)

fig, axes = plt.subplots(3, 4, figsize=(14, 6))
for ax, frame in zip(axes.flat, scene["frames"]):
    image = Image.open(DATA_DIR / f"{frame['file_path']}.png")
    ax.imshow(image)
    ax.set_title(frame["camera_name"])
    ax.axis("off")
plt.tight_layout()


## Direct `gsplat` baseline

This calls `gsplat.rendering.rasterization(...)` through `douglas_3dsplat.baseline_gsplat.train_baseline`. It needs CUDA, so on a local Mac it will usually print the Modal command instead of training.


In [ ]:
import torch

try:
    import gsplat
    print("gsplat:", getattr(gsplat, "__version__", "installed"))
except Exception as exc:
    gsplat = None
    print("gsplat is not importable in this kernel:", exc)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# Colab/Drive hotfix: patch the baseline source before importing train_baseline.
# This fixes gsplat versions that expect backgrounds.shape == (3,), not (num_cameras, 3).
import importlib
import sys

baseline_path = PACKAGE_ROOT / "douglas_3dsplat" / "baseline_gsplat.py"
old_line = "backgrounds = torch.ones(viewmats.shape[0], 3, device=viewmats.device)"
new_line = "backgrounds = torch.ones(3, device=viewmats.device)"

if baseline_path.exists():
    text = baseline_path.read_text()
    if old_line in text:
        baseline_path.write_text(text.replace(old_line, new_line))
        print("patched:", baseline_path)
    elif new_line in text:
        print("already patched:", baseline_path)
    else:
        print("background line not found in:", baseline_path)

    sys.modules.pop("douglas_3dsplat.baseline_gsplat", None)
    importlib.invalidate_caches()
else:
    print("baseline file not found:", baseline_path)
    print("If you copied functions directly into Colab cells, change the backgrounds line in that cell too.")


In [ ]:
STEPS = 300
NUM_GAUSSIANS = 1500
DOWNSCALE = 4

if torch.cuda.is_available() and gsplat is not None:
    from douglas_3dsplat.baseline_gsplat import train_baseline

    metrics = train_baseline(
        data_dir=DATA_DIR,
        output_dir=LOCAL_OUTPUT_DIR,
        steps=STEPS,
        num_gaussians=NUM_GAUSSIANS,
        downscale=DOWNSCALE,
    )
    print(metrics)
else:
    print("No CUDA gsplat kernel in this notebook kernel. Run the Modal baseline instead:")
    print(
        "modal run douglas_3dsplat/modal_setup.py "
        f"--train-baseline --steps {STEPS} "
        f"--num-gaussians {NUM_GAUSSIANS} --downscale {DOWNSCALE}"
    )


## Modal GPU baseline

Run this from the repository root when you want the baseline to train on Modal. The result is written to the Modal volume `douglas-3dsplat-output` under `gsplat_baseline_frame_0000/`.


In [ ]:
# Uncomment to launch from inside the notebook, or run the same command in your terminal.
# !modal run douglas_3dsplat/modal_setup.py --train-baseline --steps 300 --num-gaussians 1500 --downscale 4


## Inspect local outputs

If you ran the baseline in this notebook kernel, this shows the target grid and rendered grid. For Modal outputs, fetch or inspect the volume first.


In [ ]:
if (LOCAL_OUTPUT_DIR / "render_grid.png").exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(Image.open(LOCAL_OUTPUT_DIR / "targets_grid.png"))
    axes[0].set_title("targets")
    axes[0].axis("off")
    axes[1].imshow(Image.open(LOCAL_OUTPUT_DIR / "render_grid.png"))
    axes[1].set_title("gsplat render")
    axes[1].axis("off")
    plt.tight_layout()
else:
    print("No local render yet:", LOCAL_OUTPUT_DIR / "render_grid.png")
